# 05 - Google ADK Agent + Phoenix Tracing

本筆記本示範如何使用 Google ADK 建立 Agent，並搭配 Arize Phoenix 進行 tracing。

## 涵蓋內容

1. **Phoenix Tracing 設定** — 透過 OpenTelemetry 發送 traces 到 Phoenix
2. **建立 ADK Agent** — 使用 `create_agent()` 搭配 `LLMService`
3. **定義工具** — 為 Agent 建立自定義工具函式
4. **執行 Agent** — 同步/非同步執行並查看結果
5. **在 Phoenix UI 查看 Traces** — 分析 Agent 的執行流程

> **前置條件**：
> - `uv sync --group adk --group observability` 安裝 ADK 和 Phoenix 相依套件
> - 啟動 Phoenix：`docker compose up phoenix` 或 `python -m phoenix.server.main serve`
> - 在 `llm_config.yaml` 中設定正確的 LLM API

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

print("匯入完成。")

## 設定 Phoenix Tracing

Phoenix 使用 OpenTelemetry 收集 traces，支援自動 instrument LiteLLM、LangChain 等框架。

啟動 Phoenix 後，預設在 `http://localhost:6006` 提供 UI，
`http://localhost:6006/v1/traces` 接收 OTLP traces。

In [ ]:
# 設定 Phoenix tracing（透過 OpenTelemetry）
PHOENIX_ENDPOINT = "http://localhost:6006/v1/traces"
PROJECT_NAME = "adk-agent-demo"

os.environ["PHOENIX_COLLECTOR_HTTP_ENDPOINT"] = PHOENIX_ENDPOINT
os.environ["PHOENIX_PROJECT_NAME"] = PROJECT_NAME

try:
    from phoenix.otel import register
    tracer_provider = register(
        project_name=PROJECT_NAME,
        auto_instrument=True,  # 自動 instrument LiteLLM、LangChain 等
    )
    print(f"Phoenix tracing 已啟用，project: {PROJECT_NAME}")
    print("Phoenix UI: http://localhost:6006")
except Exception as e:
    print(f"[警告] Phoenix 設定失敗：{e}")
    print("請確認已安裝 arize-phoenix 和 openinference-instrumentation")
    print("安裝：uv sync --group observability")

## 建立 Google ADK Agent

使用 `app.agents.create_agent()` 建立 Agent，
透過 `LLMService` 自動從 `llm_config.yaml` 載入設定並建立 LiteLlm model。

### 架構

```
llm_config.yaml → LLMService() → create_agent(service=...) → ADK Agent
```

In [ ]:
from app.agents import create_agent, run_agent_sync
from llm_service import LLMService

service = LLMService()

# 建立最基本的 Agent（無工具）
simple_agent = create_agent(
    name="simple_assistant",
    instruction="你是一個友善的中文助手，用繁體中文回答所有問題。",
    service=service,
)

print(f"Agent 建立完成：{simple_agent.name}")
print(f"Model: {simple_agent.model}")

In [ ]:
# 執行 Agent
try:
    result = run_agent_sync(simple_agent, "什麼是 Google ADK？請簡要說明。")
    print("=== Agent 回應 ===")
    print(result)
except Exception as e:
    print(f"[警告] Agent 執行失敗：{e}")
    print("請確認 LLM 服務已啟動，且 llm_config.yaml 設定正確。")

## 建立帶工具的 Agent

ADK Agent 可以使用 Python 函式作為工具。
函式的 docstring 會作為工具描述，供 Agent 判斷何時呼叫。

In [ ]:
from datetime import datetime


def get_current_time() -> dict:
    """取得目前的日期和時間。當使用者詢問現在幾點或今天日期時使用此工具。"""
    now = datetime.now()
    return {
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "weekday": ["週一", "週二", "週三", "週四", "週五", "週六", "週日"][now.weekday()],
    }


def search_knowledge_base(query: str) -> dict:
    """搜尋內部知識庫。當使用者問到公司相關資訊時使用此工具。

    Args:
        query: 搜尋關鍵字
    """
    # 模擬知識庫搜尋
    knowledge = {
        "退款": "退款政策：購買後 30 天內可全額退款，需提供訂單編號。",
        "營業": "營業時間：週一至週五 9:00-18:00，週末休息。",
        "聯絡": "聯絡方式：客服信箱 support@example.com，電話 02-1234-5678。",
    }
    for key, value in knowledge.items():
        if key in query:
            return {"found": True, "content": value}
    return {"found": False, "content": f"找不到與 '{query}' 相關的資訊。"}


def calculate(expression: str) -> dict:
    """計算數學表達式。當使用者需要進行數學運算時使用此工具。

    Args:
        expression: 要計算的數學表達式，例如 '2 + 3 * 4'
    """
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return {"error": "不支援的運算符號"}
        result = eval(expression)
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}


print("定義了 3 個工具：get_current_time, search_knowledge_base, calculate")

In [ ]:
# 建立帶工具的 Agent
tool_agent = create_agent(
    name="tool_assistant",
    instruction=(
        "你是一個企業助手，可以使用工具來回答問題。\n"
        "- 使用 get_current_time 查詢時間\n"
        "- 使用 search_knowledge_base 查詢公司資訊\n"
        "- 使用 calculate 進行數學運算\n"
        "請用繁體中文回答。"
    ),
    tools=[get_current_time, search_knowledge_base, calculate],
    service=service,
)

print(f"Agent '{tool_agent.name}' 建立完成，工具數量：{len(tool_agent.tools)}")

In [ ]:
# 測試工具呼叫
test_queries = [
    "現在幾點了？",
    "你們的退款政策是什麼？",
    "請幫我算 (15 + 27) * 3",
]

for query in test_queries:
    print(f"\n{'='*50}")
    print(f"問題：{query}")
    print(f"{'='*50}")
    try:
        result = run_agent_sync(tool_agent, query)
        print(f"回答：{result}")
    except Exception as e:
        print(f"[警告] 執行失敗：{e}")

## 直接使用 LLMService 建立 ADK Agent

如果不需要 `app.agents` 封裝，也可以直接使用 `LLMService` 搭配 ADK 建立 Agent。

In [ ]:
from llm_service import LLMService
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

# 使用 LLMService 取得 resolved config 來建立 LiteLlm model
service = LLMService()
resolved = service._resolve()

model_name = resolved.model_name
if "/" not in model_name:
    model_name = f"openai/{model_name}"

headers = {**resolved.extra_headers}
if resolved.api_key:
    headers["Authorization"] = f"Bearer {resolved.api_key}"

model = LiteLlm(model=model_name, api_base=resolved.api_base, extra_headers=headers)

# 直接用 ADK Agent API
agent = Agent(
    name="direct_agent",
    model=model,
    instruction="你是一個助手。",
)

print(f"直接建立的 Agent: {agent.name}")
print(f"Model: {agent.model}")

## 搭配 MLflow Tracing（替代方案）

除了 Phoenix，也可以使用 MLflow 作為 tracing backend。
兩者可以同時使用，traces 會分別發送到各自的 backend。

```python
# 同時使用 MLflow + Phoenix
from app.logger import init_mlflow
init_mlflow(cfg)  # MLflow autolog

from phoenix.otel import register
register(project_name="my-project", auto_instrument=True)  # Phoenix OTEL
```

## 在 Phoenix UI 查看 Traces

執行上述範例後，可以在 Phoenix UI 中查看 Agent 的完整執行流程：

1. 開啟瀏覽器前往 `http://localhost:6006`
2. 選擇 project `adk-agent-demo`
3. 查看每次 Agent 執行的 trace，包含：
   - LLM 呼叫（輸入/輸出、token 用量）
   - 工具呼叫（參數、回傳值）
   - 整體延遲與執行流程

### docker-compose 啟動 Phoenix

```bash
# 啟動 Phoenix + MLflow
docker compose up -d

# 或只啟動 Phoenix
docker compose up phoenix -d
```

### Phoenix 服務端口

| 服務 | 端口 | 用途 |
|------|------|------|
| Phoenix UI | 6006 | Web 介面 |
| OTLP gRPC | 4317 | OpenTelemetry traces 收集 |

## 小結

| 元件 | 說明 |
|------|------|
| `LLMService` | 從 `llm_config.yaml` 載入設定，提供 LLM 呼叫能力 |
| `create_agent(service=...)` | 建立 ADK Agent，透過 LLMService 自動注入 model |
| `run_agent_sync()` | 同步執行 Agent |
| `phoenix.otel.register()` | 啟用 Phoenix OTEL tracing |
| `auto_instrument=True` | 自動追蹤 LiteLLM、LangChain 等框架 |

### 相關資源

- `01_hello_world.ipynb`：基本 LLM 呼叫 + MLflow Autolog
- `02_custom_workflow.ipynb`：LangGraph Workflow
- `docker-compose.yml`：一鍵啟動 MLflow + Phoenix